In [ ]:
import importlib
import torch
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import matplotlib as mpl

# Set global tick font size
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12

model_name_baseline = "2D_baseline_to_limited_or"
model_name_or = "2D_limited_overreporting"
relative_path = os.path.join('..', '..', 'dptorch')

notebook_dir = os.getcwd()
absolute_path = os.path.abspath(os.path.join(notebook_dir, relative_path))

sys.path.insert(0, absolute_path)

def _iter_num(p):
    try:
        return int(p.stem.split("_")[-1])
    except ValueError:
        return -1
import pathlib

list_all_Iter_bl=list(pathlib.Path(os.path.abspath(f"data/2D/baseline")).rglob("*.pth"))
all_checkpoint_iters_bl = [_iter_num(p) for p in list_all_Iter_bl]
list_all_Iter_or=list(pathlib.Path(os.path.abspath(f"data/2D/overreporting")).rglob("*.pth"))
all_checkpoint_iters_or = [_iter_num(p) for p in list_all_Iter_or]

#### what to load
checkpoint_file_bl = max(all_checkpoint_iters_bl)
checkpoint_file_or = max(all_checkpoint_iters_or)

last_saved_checkpoint_file_bl = 5939
last_saved_checkpoint_file_or = 4379


model_bl = importlib.import_module(f"{model_name_baseline}.Model")
model_or = importlib.import_module(f"{model_name_or}.Model")

# RNG
torch.manual_seed(123)

m = model_bl.SpecifiedModel.load(
    path=os.path.abspath(f"data/2D/baseline/Iter_{checkpoint_file_bl}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_baseline},
)

m_or = model_or.SpecifiedModel.load(
    path=os.path.abspath(f"data/2D/overreporting/Iter_{checkpoint_file_or}.pth"),
    cfg_override={"distributed": False, "init_with_zeros": False, "MODEL_NAME": model_name_or},
)




In [ ]:
points = np.loadtxt((f"data/2D/baseline/simulation_{checkpoint_file_bl}.txt"))
disc_states = (points[:,2]).astype(np.int64)
mask_0 = disc_states == 0
mask_1 = disc_states == 1
points_0 = points[mask_0]
points_1 = points[mask_1]

promise_util = np.zeros((points.shape[0]-1,))
consumption = np.zeros((points.shape[0]-1,))
util = np.zeros((points.shape[0]-1,))
indx_c = m.P["c_1"]
indx_u = m.P["u_1"]
run_in_time = 100
disc_states_ = disc_states[run_in_time:-1]
mask_hist_0 = disc_states_ == 0
mask_hist_1 = disc_states_ == 1
for indx in range(points.shape[0]-1):
    promise_util[indx] = points[indx+1,disc_states[indx]]
    consumption[indx] = points[indx+1,7 + indx_c+ disc_states[indx]]
    util[indx] = points[indx+1,7 + indx_u + disc_states[indx]]

In [ ]:
points_or = np.loadtxt((f"data/2D/overreporting/simulation_{checkpoint_file_or}.txt"))
disc_states = (points_or[:,2]).astype(np.int64)
mask_0 = disc_states == 0
mask_1 = disc_states == 1
points_0 = points_or[mask_0]
points_1 = points_or[mask_1]

promise_util_or = np.zeros((points_or.shape[0]-1,))
consumption_or = np.zeros((points_or.shape[0]-1,))
util_or = np.zeros((points_or.shape[0]-1,))
indx_c = m_or.P["c_1"]
indx_u = m_or.P["u_1"]
upper_c = m_or.cfg["model"]["params"]["upper_trans"]
run_in_time = 100
disc_states_ = disc_states[run_in_time:-1]
mask_hist_0 = disc_states_ == 0
mask_hist_1 = disc_states_ == 1
for indx in range(points_or.shape[0]-1):
    promise_util_or[indx] = points_or[indx+1,disc_states[indx]]
    consumption_or[indx] = np.minimum(upper_c,points_or[indx+1,7 + indx_c+ disc_states[indx]])
    util_or[indx] = points_or[indx+1,7 + indx_u + disc_states[indx]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Adjust spacing between plots
fig.tight_layout(pad=4.0)

# Plotting the histogram
axes[0].hist(promise_util[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
axes[0].hist(promise_util_or[run_in_time:], bins=15, edgecolor='black',label="over-reporting", alpha=0.5)

# Adding labels and title
axes[0].set_xlabel('Promise utility', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Frequency', fontsize=14, fontweight='bold')
# plt.title('Histogram of NumPy Vector')
axes[0].legend(fontsize=12)

# Plotting the histogram
axes[1].hist(consumption[run_in_time:], bins=10, edgecolor='black',label="baseline", alpha=0.5)
axes[1].hist(consumption_or[run_in_time:], bins=10, edgecolor='black',label="over-reporting", alpha=0.5)

# Adding labels and title
axes[1].set_xlabel('Consumption', fontsize=14, fontweight='bold')
# axes[1].set_ylabel('Frequency', fontsize=14)
# plt.legend()

# Display the plot
plt.savefig('Figure_6_compare_long_run_histogram.pdf', dpi=500, bbox_inches='tight')


### BAL

In [ ]:
def bayesian_opt_criterion(m, eval_pt,discrete_state, target_p, rho, beta):

    # #compute bayesian optimization criteria
    mean_v = m.M[discrete_state][target_p].predict_mean(
                    eval_pt
                )
    var_v = m.M[discrete_state][target_p].predict_var(
                    eval_pt
                )

    #Deisenroth criterion
    out_vec = (rho * (mean_v) + beta / 2.0 * torch.log(var_v + 1e-15))

    return out_vec

In [ ]:
bal_util = 0
n_types = m.cfg["model"]["params"]["n_types"]

for indxt in range(n_types):
    mask_0 = m.state_sample[:,-1] == indxt
    no_init_samples = m.cfg["no_samples"]
    n_pts = m.state_sample[mask_0,:].shape[0]
    beta =1
    rho=1

    indxp = n_pts-no_init_samples - 1

    train_sample = m.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m.V_sample[mask_0][:no_init_samples+indxp]
    m.M[indxt][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util += bayesian_opt_criterion(m,m.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],indxt,0,rho,beta) / n_types

In [ ]:
bal_util

In [ ]:
bal_util_or = 0
n_types = m_or.cfg["model"]["params"]["n_types"]

for indxt in range(n_types):
    mask_0 = m_or.state_sample[:,-1] == indxt
    no_init_samples = m_or.cfg["no_samples"]
    n_pts = m_or.state_sample[mask_0,:].shape[0]
    beta =1
    rho=1

    indxp = n_pts-no_init_samples - 1

    train_sample = m_or.state_sample[mask_0,:][:no_init_samples+indxp,:-1]
    train_v = m_or.V_sample[mask_0][:no_init_samples+indxp]
    m_or.M[indxt][0].set_train_data(
        train_sample,
        train_v,
        strict=False,
    )

    bal_util_or += bayesian_opt_criterion(m_or,m_or.state_sample[mask_0,:][no_init_samples+indxp:no_init_samples+indxp+1,:-1],indxt,0,rho,beta) / n_types

In [ ]:
bal_util_or

### Errors

In [ ]:
error_data_bl = np.loadtxt(f"data/2D/baseline/V_func_error_{checkpoint_file_bl}.txt")
L2_bl = error_data_bl[:, 2]
Linf_bl = error_data_bl[:, 3]

error_data_or = np.loadtxt(f"data/2D/overreporting/V_func_error_{checkpoint_file_or}.txt")
L2_or = error_data_or[:, 2]
Linf_or = error_data_or[:, 3]

sim_data_bl = np.loadtxt(f"data/2D/baseline/simulation_{checkpoint_file_bl}.txt")
sim_data_or = np.loadtxt(f"data/2D/overreporting/simulation_{checkpoint_file_or}.txt")

sim_diff_bl = sim_data_bl[:, 6]
sim_diff_or = sim_data_or[:, 6]


sim_L2_bl = np.sqrt(np.mean(sim_diff_bl**2))
sim_Linf_bl = np.max(sim_diff_bl)
sim_L2_or = np.sqrt(np.mean(sim_diff_or**2))
sim_Linf_or = np.max(sim_diff_or)

def format_latex_sci(value):
    mantissa, exponent = f"{value:.1e}".split("e")
    return rf"${float(mantissa):.1f}\cdot 10^{{{int(exponent)}}}$"


latex_table = f"""

\\begin{{tabular}}{{l|l|c|c}}
    \\hline \\hline
    \\text{{Model Version}} & \\text{{Error type}} & \\text{{$L_2$}} & \\text{{$L_\\infty$}} \\\\
    \\hline \\hline
    Benchmark & Criterion 2 (global error) & {{{format_latex_sci(L2_bl[-1])}}} & {{{format_latex_sci(Linf_bl[-1])}}} \\\\
    Benchmark & Criterion 3 (error along a simulated path) & {{{format_latex_sci(sim_L2_bl)}}} & {{{format_latex_sci(sim_Linf_bl)}}} \\\\
    \\hline
    Overreporting & Criterion 2 (global error) & {{{format_latex_sci(L2_or[-1])}}} & {{{format_latex_sci(Linf_or[-1])}}} \\\\
    Overreporting & Criterion 3 (error along a simulated path) & {{{format_latex_sci(sim_L2_or)}}} & {{{format_latex_sci(sim_Linf_or)}}} \\\\
    \\hline

\\end{{tabular}}

""".strip()


print(latex_table)

print(latex_table)

with open("Table_7_error_table.tex", "w", encoding="utf-8") as table_file:
    table_file.write(latex_table)

In [ ]:
m_l2 = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l2"]).numpy()))
m_inf = float(np.mean((m.metrics[list(m.metrics.keys())[-1]]["l_inf"]).numpy()))
mor_l2 = float(np.mean((m_or.metrics[list(m_or.metrics.keys())[-1]]["l2"]).numpy()))
mor_inf = float(np.mean((m_or.metrics[list(m_or.metrics.keys())[-1]]["l_inf"]).numpy()))

table_path = os.path.join(notebook_dir, "2D_pointwise_error_table.tex")
with open(table_path, "w", encoding="utf-8") as f:
    f.write(
        "\\begin{table}[ht]\n"
        "\\centering\n"
        "\\begin{tabular}{lcc}\n"
        "\\hline\n"
        "2D Model & $L_2$ & $L_\\infty$ \\\\\n"
        "\\hline\n"
        f"Baseline & ${m_l2:.4e}$ & ${m_inf:.4e}$ \\\\\n"
        f"Overreporting & ${mor_l2:.4e}$ & ${mor_inf:.4e}$ \\\\\n"
        "\\hline\n"
        "\\end{tabular}\n"
        "\\end{table}\n"
    )

print(f"Saved LaTeX table to: {table_path}")